# ATML PA2: Task 5 - Feedback Source: RLVR versus RLAIF

This notebook runs all Task 5 feedback source experiments on a Kaggle GPU environment:
1. **Environment & Assets Setup** (clones repo, installs dependencies, downloads course assets)
2. **Step 1: In-Domain Comparison on GSM8K** (`sft`, `rlvr`, `rlaif`)
   - Exact accuracy, format compliance, response length, AI pairwise win rate vs SFT, and verifier-judge agreement.
3. **Step 2: Controlled Reward-Sensitivity & Robustness Study** (100 diagnostic examples across 20 problems)
   - Evaluates reasoning corruption, wrong final answer, persuasive filler, gold distractor, and trade-offs.
   - Computes $S_{\text{reason}}$ and $S_{\text{outcome}}$, better-response rates, tie rates, and wrong-preference rates.
4. **Step 3: Out-of-Domain SVAMP Transfer Evaluation** (100 fixed SVAMP examples)
   - Evaluates generalization drop, format retention, response length, and transfer win rates.
5. **Step 4: Cross-Task Synthesis & Qualitative Case Extraction**
   - Compares feedback coverage, noise, exploitability, and inference cost.
6. **Results Packaging & Inspection** (`task5_results.zip`)

## 1. Verify GPU and Environment

In [ ]:
!nvidia-smi

## 2. Clone Your GitHub Repository
Ensure the latest code is pulled from your repository.

In [ ]:
REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA2-LLM-PostTraining.git"

import os
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    !git clone $REPO_URL
    %cd $repo_name
else:
    %cd $repo_name
    !git fetch origin
    !git reset --hard origin/main
    !git pull

!git status

## 3. Install Dependencies & Validate Course Assets

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Step 1: In-Domain GSM8K Math Evaluation
Evaluates **SFT**, **RLVR** (`checkpoints/rlvr_policy`), and **RLAIF** (`checkpoints/rlaif_policy`) on the fixed GSM8K evaluation subset (300 problems).
- Generates deterministic responses (`temperature=0.0`, `max_new_tokens=512`).
- Computes exact final-answer accuracy, format compliance (`#### <number>`), and response length.
- Runs Pairwise AI Judge (`Qwen/Qwen2.5-3B-Instruct`) for pairwise win rates against SFT and head-to-head.
- Computes verifier-judge decisive agreement.

In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm --batch-size 16

## 5. Step 2: Controlled Reward Sensitivity & Robustness Study
Scores the 100-response diagnostic set (20 problems × 5 perturbation variants) under both the **Exact Verifier** and the **AI Pairwise Judge**:
1. **Reasoning Corruption**: Tests $S_{\text{reason}} = \Pr[R(y_{\text{clean}}) > R(y_{\text{corrupt}})]$ (same correct answer, degraded reasoning).
2. **Outcome Corruption**: Tests $S_{\text{outcome}} = \Pr[R(y_{\text{correct}}) > R(y_{\text{wrong}})]$ (good reasoning, wrong final answer).
3. **Persuasive Filler**: Tests verbosity / persuasive framing bias.
4. **Gold Distractor**: Tests whether mentioning gold number as distractor misleads the verifier parser or judge.
5. **Trade-Off**: Corrupt reasoning with correct answer vs good reasoning with wrong answer.

In [ ]:
!python -m task5_feedback.score_perturbations --config configs/feedback.yaml

## 6. Step 3: Out-of-Domain SVAMP Transfer Evaluation
Evaluates SFT, RLVR, and RLAIF on the fixed 100-example SVAMP challenge transfer subset (`data/math_transfer_eval.jsonl`) without retraining.
Measures transfer accuracy, win rate against SFT under the AI judge, format retention, and performance drops.

In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer --batch-size 16

## 7. Step 4: Comprehensive Synthesis & Domain Transfer Analysis
Combines in-domain GSM8K, out-of-domain SVAMP, and diagnostic perturbation results into final comparative tables and qualitative disagreement cases.

In [ ]:
!python -m task5_feedback.compare_feedback --config configs/feedback.yaml

## 8. View and Inspect All Task 5 Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

res_dir = Path("results/task5_feedback")

print("=== IN-DOMAIN VS TRANSFER DOMAIN COMPARISON TABLE ===")
final_table_csv = res_dir / "final_comparison_table.csv"
if final_table_csv.exists():
    display(pd.read_csv(final_table_csv))

print("\n=== CONTROLLED PERTURBATION STUDY SUMMARY ===")
perturb_csv = res_dir / "perturbation_scores.csv"
if perturb_csv.exists():
    display(pd.read_csv(perturb_csv))

print("\n=== SENSITIVITY METRICS (S_reason and S_outcome) ===")
perturb_json = res_dir / "perturbation_scores.json"
if perturb_json.exists():
    pj = json.loads(perturb_json.read_text())
    print("S_reason:", pj.get("s_reason"))
    print("S_outcome:", pj.get("s_outcome"))

print("\n=== QUALITATIVE DIAGNOSTIC CASES (Sample Disagreements) ===")
cases_json = res_dir / "qualitative_diagnostic_cases.json"
if cases_json.exists():
    cases = json.loads(cases_json.read_text())
    print(f"Found {len(cases)} diagnostic case(s). First 2 cases:")
    print(json.dumps(cases[:2], indent=2))


## 9. Package Results for Download
Zips all generated JSON/CSV results and generated response files from `results/task5_feedback` so you can download them directly from Kaggle.

In [ ]:
!zip -r task5_results.zip results/task5_feedback
print("task5_results.zip ready in Kaggle working directory!")

# Optional: push results back to your GitHub repository
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"
# !git add results/task5_feedback
# !git commit -m "Add Task 5 RLVR vs RLAIF evaluation results"
# !git push